In [ ]:
!pip install -q sentence-transformers pandas tqdm openpyxl

import json
import os
import re
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

# ═══════════════════════════════════════════════════════════════════════════
# CẤU HÌNH ĐƯỜNG DẪN DATASET VIỆT NAM (ĐỒNG BỘ)
# ═══════════════════════════════════════════════════════════════════════════
DATASET_ROOT = "/kaggle/input/datasets/thaiphonghuan/scoring/vietname-dataset-scoring - training"
CV_DIR = os.path.join(DATASET_ROOT, "cv")
JD_DIR = os.path.join(DATASET_ROOT, "jd")
TEST_MATCH_CSV = os.path.join(DATASET_ROOT, "dataset/test/match/test_match.csv")
TEST_MISMATCH_CSV = os.path.join(DATASET_ROOT, "dataset/test/mismatch/test_mismatch.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Sử dụng model bge như các bản multi-view trước
MODEL_NAME = 'bkai-foundation-models/vietnamese-bi-encoder'
sbert_model = SentenceTransformer(MODEL_NAME).to(device)

def clean_text(text):
    """Tiền xử lý văn bản thô"""
    if not text: return ""
    text = re.sub(r'[\n\r/]', ' ', text)
    text = text.replace('*', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

In [ ]:
def get_holistic_text(json_data):
    """Gộp tất cả các View trong JSON thành một khối văn bản duy nhất."""
    combined_parts = []
    # Duyệt qua toàn bộ các giá trị trong JSON
    for val in json_data.values():
        if val is None or str(val).lower() == "missing_section":
            continue
        if isinstance(val, list):
            combined_parts.append(" ".join([str(v) for v in val]))
        else:
            combined_parts.append(str(val))
    
    return clean_text(" ".join(combined_parts))

def run_single_view_on_csv():
    all_scores = []
    y_true_reg = []
    y_true_bin = []
    
    configs = [(TEST_MATCH_CSV, 1), (TEST_MISMATCH_CSV, 0)]
    
    print(f"🚀 Đang chạy Single-view Holistic Matching cho dữ liệu Việt Nam...")
    for csv_path, b_label in configs:
        if not os.path.exists(csv_path):
            print(f"⚠️ Bỏ qua: {csv_path}")
            continue
            
        df = pd.read_csv(csv_path)
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"CSV {b_label}"):
            # 1. Load JSON (Xử lý BOM và Fallback)
            with open(os.path.join(CV_DIR, row['cv']), 'r', encoding='utf-8-sig') as f:
                cv_obj = json.load(f)
                cv_j = cv_obj.get('resume_structured', cv_obj)
            with open(os.path.join(JD_DIR, row['jd']), 'r', encoding='utf-8-sig') as f:
                jd_obj = json.load(f)
                jd_j = jd_obj.get('job_description_structured', jd_obj)

            # 2. Gộp văn bản thành "Single View"
            cv_text = get_holistic_text(cv_j)
            jd_text = get_holistic_text(jd_j)
            
            with torch.no_grad():
                # 3. Mã hóa mỗi tài liệu thành DUY NHẤT 1 vector
                cv_emb = sbert_model.encode(cv_text, convert_to_tensor=True)
                jd_emb = sbert_model.encode(jd_text, convert_to_tensor=True)
                
                # 4. Tính Cosine Similarity
                score = util.cos_sim(cv_emb, jd_emb).item()
            
            all_scores.append(score)
            y_true_reg.append(float(row['score']))
            y_true_bin.append(b_label)
            
    return np.array(all_scores), np.array(y_true_reg), np.array(y_true_bin)

# Thực thi
y_pred_test, y_true_reg, y_true_bin = run_single_view_on_csv()

In [ ]:
import openpyxl

threshold = 0.5
y_pred_bin = (y_pred_test >= threshold).astype(int)

# Tính toán Metrics
mae = mean_absolute_error(y_true_reg, y_pred_test)
mse = mean_squared_error(y_true_reg, y_pred_test)
acc = accuracy_score(y_true_bin, y_pred_bin)
pre = precision_score(y_true_bin, y_pred_bin, zero_division=0)
rec = recall_score(y_true_bin, y_pred_bin, zero_division=0)
f1  = f1_score(y_true_bin, y_pred_bin, zero_division=0)

print("\n" + "═"*60)
print("🏆 KẾT QUẢ SINGLE-VIEW HOLISTIC BASELINE")
print("═"*60)
print(f"MAE: {mae:.4f} | MSE: {mse:.4f}")
print(f"Accuracy: {acc*100:.2f}%")
print(f"Precision: {pre:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
print("═"*60)

# Cập nhật kết quả vào Excel
def upsert_baseline_excel(path, model_desc, metrics):
    header = ["Encoder", "MAE", "MSE", "Accuracy", "Precision", "Recall", "F1-Score"]
    if not os.path.exists(path):
        wb = openpyxl.Workbook(); ws = wb.active; ws.title = "baselines"; ws.append(header)
    else:
        wb = openpyxl.load_workbook(path)
        ws = wb["baselines"] if "baselines" in wb.sheetnames else wb.create_sheet("baselines")
    
    ws.append([model_desc] + [round(float(m), 4) for m in metrics])
    wb.save(path)
    print(f"💾 Đã cập nhật kết quả Single-view vào: {path}")

EXCEL_PATH = "/kaggle/working/results_baselines_vn.xlsx"
upsert_baseline_excel(EXCEL_PATH, f"{MODEL_NAME}_single_view_holistic", [mae, mse, acc, pre, rec, f1])